# 01 · Brandimarte örneği, CP-SAT ve bağımsız doğrulama

Bu notebook'ta Atakan'ın WP-1A hattını adım adım çalıştırıyoruz:

1. Sabit commit'teki kaynak depoyu kontrol etmek (`check_upstream`)
2. Mk01'i **sıkı girdi denetimiyle** okumak (`read_instance`)
3. Basit referans yöntem (`baseline`, iş sırası + en erken bitiş) ile CP-SAT'ı (`solve`) karşılaştırmak
4. Çizelgeyi çözücüden bağımsız doğrulamak (`core.validation.validate`)
5. Plotly Gantt ile göstermek

> Süreler **benchmark zaman birimidir**, dakika değildir.

**Önkoşul (bir kez):** repo kökünde `sh scripts/fetch_upstream.sh` kaynak depoyu sabit commit'e çeker;
ardından README'deki ortam kurulumu. Notebook'lar yalnızca repodaki kodu çağırır, yeni çözücü/doğrulayıcı yazmaz.

In [1]:
# Notebook'u tutorials/ veya repo kökünden açabilirsiniz; importlar repo kökünden çalışır.
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "tutorials":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import plotly.io as pio
pio.renderers.default = "notebook_connected"   # plotly.js'i her notebook'a gömmez; görüntülemek için internet gerekir
print("Repo kökü:", ROOT)

Repo kökü: /Users/nazlsu/Desktop/PickWise


In [2]:
from core.cpsat_solver import UPSTREAM, check_upstream, read_instance, baseline, solve
from core.validation import validate

print("Kaynak commit:", check_upstream())

Kaynak commit: d088a26582f5cb79c6b4f2ad94f68ac7c50878cb


## 1. Örneği okumak

`read_instance` dosyayı önce kendisi token token denetler, sonra kaynak parser ile okur ve iki sonucu karşılaştırır.
Boyutlar dosyadan okunur; elle yazılmaz.

In [3]:
path = UPSTREAM / "data/fjsp/brandimarte/Mk01.fjs"
env, jobs, n_machines = read_instance(path)

print("İş sayısı      :", len(jobs))
print("Makine sayısı  :", n_machines)
print("Operasyon sayısı:", sum(len(job) for job in jobs))
print("İş 1, operasyon 1 seçenekleri (0 tabanlı makine → süre):", jobs[0][0])

İş sayısı      : 10
Makine sayısı  : 6
Operasyon sayısı: 55
İş 1, operasyon 1 seçenekleri (0 tabanlı makine → süre): {0: 5, 2: 4}


Kullanıcıya makineler 1 tabanlı gösterilir: `{0: 5, 2: 4}` → **M1'de 5 veya M3'te 4 birim**.

Bozuk bir dosya sessizce kabul edilmez. Aşağıda var olmayan bir makine (M9) içeren küçük bir dosya deniyoruz:

In [4]:
import tempfile

bad = Path(tempfile.mkdtemp()) / "bozuk.fjs"
bad.write_text("1 2 1\n1 1 9 5\n")   # 1 iş, 2 makine; operasyon M9'a atanmış
try:
    read_instance(bad)
except ValueError as error:
    print("Reddedildi:", error)

Reddedildi: Invalid machine/duration


## 2. Referans yöntem ve CP-SAT

In [5]:
reference = baseline(jobs, n_machines)
cpsat = solve(env, time_limit=10)

print(f"Referans (iş sırası + en erken bitiş): makespan {reference['makespan']}")
print(f"CP-SAT: durum {cpsat['status']}, makespan {cpsat['makespan']}")
print("CP-SAT sonucundaki alanlar:", sorted(cpsat))

Horizon = 254
Referans (iş sırası + en erken bitiş): makespan 87
CP-SAT: durum OPTIMAL, makespan 40
CP-SAT sonucundaki alanlar: ['best_bound', 'makespan', 'runtime_seconds', 'schedule', 'seed', 'status', 'time_limit_seconds', 'workers']


**Durumları doğru okumak:** `OPTIMAL` bu model için optimum kanıtıdır. `FEASIBLE` geçerli plan bulundu ama optimum
kanıtlanmadı demektir. `UNKNOWN` süre içinde çözüm/kanıt yok demektir; imkânsızlık kanıtı değildir.
CP-SAT tek worker ve `seed=0` ile çalışır; farklı donanımda sonuç değişebilir.

## 3. Bağımsız doğrulama

In [6]:
for name, plan in [("Referans", reference), ("CP-SAT", cpsat)]:
    errors = validate(jobs, n_machines, plan["schedule"])
    print(f"{name}: {'geçerli' if not errors else errors}")

Referans: geçerli
CP-SAT: geçerli


Doğrulayıcının hatayı gerçekten yakaladığını görmek için CP-SAT çizelgesinin bir kopyasını bozuyoruz:

In [7]:
import copy

broken = copy.deepcopy(cpsat["schedule"])
first = min(broken, key=lambda r: r["start"])
other = next(r for r in broken if r["machine"] == first["machine"] and r is not first)
shift = other["start"] - first["start"]
other["start"] -= shift
other["end"] -= shift
print(validate(jobs, n_machines, broken))

['Overlap: machine 1']


## 4. Gantt

In [8]:
import plotly.graph_objects as go

def gantt(rows, title, machine_label, job_label, x_title):
    """rows: çizelge satırları (job, operation, machine, start, end)."""
    fig = go.Figure()
    for j in sorted({r["job"] for r in rows}):
        group = [r for r in rows if r["job"] == j]
        fig.add_trace(go.Bar(
            name=job_label(j), orientation="h",
            y=[machine_label(r["machine"]) for r in group],
            x=[r["end"] - r["start"] for r in group],
            base=[r["start"] for r in group],
            text=[f"O{r['operation'] + 1}" for r in group],
            hovertemplate="%{y}<br>Başlangıç: %{base}<br>Süre: %{x}<extra>%{fullData.name}</extra>",
        ))
    fig.update_layout(title=title, barmode="overlay", xaxis_title=x_title,
                      yaxis_title="Makine", height=420, legend_title="İş")
    fig.update_yaxes(categoryorder="category descending")
    return fig

In [9]:
gantt(cpsat["schedule"], f"Mk01 · CP-SAT ({cpsat['status']}, makespan {cpsat['makespan']})",
      machine_label=lambda m: f"M{m + 1}", job_label=lambda j: f"İş {j + 1}",
      x_title="Benchmark zaman birimi (dakika değil)")

## Özet

- Girdi iki kez denetlenir; çözüm çözücüden bağımsız doğrulanır.
- Referans yöntem kaynak deponun FIFO/SPT uygulaması **değildir**; basit ve tekrarlanabilir bir karşılaştırma noktasıdır.
- Tüm Mk01–Mk10 için: `python -m benchmarks.brandimarte_runner --time-limit 10` → `results/brandimarte/`.